# A. SVM Classifier for MNIST Dataset

## Setup

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import os

# Machine Learning
from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# Konfigurasi plot
plt.rcParams['figure.figsize'] = (6,4)
plt.rcParams['font.size'] = 12

# Folder untuk menyimpan hasil
os.makedirs("plots", exist_ok=True)

## 2 Load Dataset MNIST

In [ ]:
# Load dataset MNIST
mnist = fetch_openml('mnist_784', version=1)

X = mnist.data
y = mnist.target.astype(int)

print("Shape data:", X.shape)
print("Shape label:", y.shape)

## 3. Visualisasi

In [ ]:
# Tampilkan beberapa contoh digit
fig, axes = plt.subplots(2,5)

for i, ax in enumerate(axes.flat):
    ax.imshow(X.iloc[i].values.reshape(28,28), cmap='gray')
    ax.set_title(f"Label: {y[i]}")
    ax.axis('off')

plt.tight_layout()
plt.savefig("plots/sample_digits.png")
plt.show()

## 4. Preprocessing & Scaling

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# (Opsional tapi disarankan) Normalisasi pixel ke 0–1
X = X / 255.0

# Gunakan subset agar training lebih cepat
X_small, _, y_small, _ = train_test_split(
    X, y,
    train_size=12000,
    stratify=y,
    random_state=42
)

# Split train-test
X_train, X_test, y_train, y_test = train_test_split(
    X_small, y_small,
    test_size=0.2,
    random_state=42
)

print("Train shape:", X_train.shape)
print("Test shape:", X_test.shape)

In [ ]:
scaler = StandardScaler()

# Fit hanya pada training data
X_train_scaled = scaler.fit_transform(X_train)

# Gunakan parameter yang sama untuk test
X_test_scaled = scaler.transform(X_test)

In [ ]:
print("Mean (train):", np.mean(X_train_scaled))
print("Std (train):", np.std(X_train_scaled))

In [ ]:
plt.hist(X_train_scaled[0], bins=30)
plt.title("Distribusi fitur setelah scaling")

plt.savefig("plots/scaling_distribution.png")
plt.show()

## 5. Training SVM

In [ ]:
from sklearn.svm import SVC

# Buat model SVM
svm_clf = SVC(
    kernel='rbf',
    C=1.0,
    gamma='scale'
)

# Training
svm_clf.fit(X_train_scaled, y_train)

In [ ]:
from sklearn.metrics import accuracy_score

# Prediksi
y_pred = svm_clf.predict(X_test_scaled)

# Evaluasi
accuracy = accuracy_score(y_test, y_pred)

print("Baseline Accuracy:", accuracy)

In [ ]:
fig, axes = plt.subplots(2,5)

for i, ax in enumerate(axes.flat):
    ax.imshow(X_test.iloc[i].values.reshape(28,28), cmap='gray')
    ax.set_title(f"Pred: {y_pred[i]}")
    ax.axis('off')

plt.tight_layout()
plt.savefig("plots/baseline_predictions.png")
plt.show()

## 6. Fine Tuning SVM (GridSearch)

In [ ]:
from sklearn.model_selection import GridSearchCV

param_grid = {
    'C': [1, 5, 10],
    'gamma': ['scale', 0.01, 0.001]
}

grid_search = GridSearchCV(
    estimator=SVC(kernel='rbf'),
    param_grid=param_grid,
    cv=3,
    verbose=2,
    n_jobs=-1
)

grid_search.fit(X_train_scaled, y_train)

In [ ]:
print("Best Parameters:", grid_search.best_params_)
print("Best CV Score:", grid_search.best_score_)

In [ ]:
best_model = grid_search.best_estimator_

In [ ]:
y_pred_tuned = best_model.predict(X_test_scaled)

accuracy_tuned = accuracy_score(y_test, y_pred_tuned)

print("Tuned Accuracy:", accuracy_tuned)

In [ ]:
print("Baseline Accuracy :", accuracy)
print("Tuned Accuracy    :", accuracy_tuned)

In [ ]:
fig, axes = plt.subplots(2,5)

for i, ax in enumerate(axes.flat):
    ax.imshow(X_test.iloc[i].values.reshape(28,28), cmap='gray')
    ax.set_title(f"Pred: {y_pred_tuned[i]}")
    ax.axis('off')

plt.tight_layout()
plt.savefig("plots/tuned_predictions.png")
plt.show()

## 7. Confusion Matrix & Error Analysis

In [ ]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

cm = confusion_matrix(y_test, y_pred_tuned)

disp = ConfusionMatrixDisplay(confusion_matrix=cm)

disp.plot()
plt.title("Confusion Matrix - Tuned SVM")

plt.savefig("plots/confusion_matrix.png")
plt.show()

In [ ]:
from sklearn.metrics import classification_report

print(classification_report(y_test, y_pred_tuned))

In [ ]:
# Cari indeks yang salah prediksi
errors = (y_pred_tuned != y_test)

X_errors = X_test[errors]
y_errors = y_test[errors]
y_pred_errors = y_pred_tuned[errors]

print("Jumlah error:", len(X_errors))

In [ ]:
fig, axes = plt.subplots(2,5)

for i, ax in enumerate(axes.flat):
    ax.imshow(X_errors.iloc[i].values.reshape(28,28), cmap='gray')
    ax.set_title(f"T:{y_errors.iloc[i]} P:{y_pred_errors[i]}")
    ax.axis('off')

plt.tight_layout()
plt.savefig("plots/error_samples.png")
plt.show()